# SAFB Centralized Experiment (Colab)

This notebook runs the new centralized SAFB pipeline in `centralized/train_safb.py`.

Goal: verify whether centralized training yields better ACC/ASR trade-off than federated runs.


In [ ]:
import torch

print('=' * 72)
print('Environment Check')
print('=' * 72)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print('=' * 72)


In [ ]:
# Clone repo (or pull latest) and enter project directory.
import os
import sys
import subprocess

REPO_URL = 'https://github.com/zyh-hnu/ANB.git'
PROJECT_DIR = '/kaggle/working/SAFB' if os.path.exists('/kaggle') else '/content/SAFB'

def run_cmd(cmd, check=True):
    """Run command using subprocess for better control."""
    print(f'[CMD] {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}')
    return result

if not os.path.exists(PROJECT_DIR):
    print(f'[INFO] Cloning repository to {PROJECT_DIR}...')
    run_cmd(['git', 'clone', REPO_URL, PROJECT_DIR])
else:
    print(f'[INFO] Project exists, pulling latest...')
    run_cmd(['git', '-C', PROJECT_DIR, 'pull'], check=False)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Working directory:', os.getcwd())


In [ ]:
# Install dependencies.
!pip -q install -r requirements.txt
!pip -q install hdbscan


In [ ]:
# Experiment configuration (centralized SAFB).
from datetime import datetime

CFG = {
    'dataset': 'CIFAR10',
    'freq_strategy': 'ANB',
    'target_label': 0,
    'epsilon': 0.1,
    'poison_rate': 0.15,
    'cross_ratio': 0.50,
    'backdoor_boost_weight': 0.20,
    'epochs': 8,
    'batch_size': 256,
    'learning_rate': 0.05,
    'train_subset': 8000,
    'test_subset': 2000,
    'num_workers': 2,
    'log_interval': 20,
    'seed': 42,
    'device': 'auto',
    'eval_multi_client_ids': '',
    'min_asr_for_tradeoff': 0.85,
    'use_phased_chaos': 1,
    'use_spectral_smoothing': 0,
    'use_freq_sharding': 1,
    'use_dual_routing': 0,
}

RUN_NAME = datetime.now().strftime('colab_centralized_%Y%m%d_%H%M%S')

print('=' * 72)
print('Centralized SAFB Config')
print('=' * 72)
for key, value in CFG.items():
    print(f'{key}: {value}')
print('run_name:', RUN_NAME)
print('=' * 72)


In [ ]:
# Launch centralized training.
import subprocess
import shlex

cmd = [
    'python', '-u', 'centralized/train_safb.py',
    '--dataset', str(CFG['dataset']),
    '--freq-strategy', str(CFG['freq_strategy']),
    '--target-label', str(CFG['target_label']),
    '--epsilon', str(CFG['epsilon']),
    '--poison-rate', str(CFG['poison_rate']),
    '--cross-ratio', str(CFG['cross_ratio']),
    '--backdoor-boost-weight', str(CFG['backdoor_boost_weight']),
    '--epochs', str(CFG['epochs']),
    '--batch-size', str(CFG['batch_size']),
    '--learning-rate', str(CFG['learning_rate']),
    '--train-subset', str(CFG['train_subset']),
    '--test-subset', str(CFG['test_subset']),
    '--num-workers', str(CFG['num_workers']),
    '--log-interval', str(CFG['log_interval']),
    '--seed', str(CFG['seed']),
    '--device', str(CFG['device']),
    '--eval-multi-client-ids', str(CFG['eval_multi_client_ids']),
    '--min-asr-for-tradeoff', str(CFG['min_asr_for_tradeoff']),
    '--use-phased-chaos', str(CFG['use_phased_chaos']),
    '--use-spectral-smoothing', str(CFG['use_spectral_smoothing']),
    '--use-freq-sharding', str(CFG['use_freq_sharding']),
    '--use-dual-routing', str(CFG['use_dual_routing']),
    '--output-dir', 'results/centralized_runs',
    '--run-name', RUN_NAME,
]

print('Command:')
print(' '.join(shlex.quote(x) for x in cmd))

subprocess.run(cmd, check=True)
RUN_DIR = f'results/centralized_runs/{RUN_NAME}'
print('\nRun directory:', RUN_DIR)


In [ ]:
# Read and print final metrics.
import json
from pathlib import Path

run_dir = Path(RUN_DIR)
summary_path = run_dir / 'summary.json'
history_path = run_dir / 'history.json'

if not summary_path.exists() or not history_path.exists():
    raise FileNotFoundError(f'Result files not found under: {run_dir}')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
history = json.loads(history_path.read_text(encoding='utf-8'))

print('=' * 72)
print('Final Metrics')
print('=' * 72)
print('Final ACC:', f"{summary['final_acc']:.2%}")
print('Final ASR:', f"{summary['final_asr']:.2%}")
if summary.get('final_asr_multi') is not None:
    print('Final ASR (multi):', f"{summary['final_asr_multi']:.2%}")
print('Best ASR:', f"{summary['best_asr']:.2%}")
print('Best-ASR ACC:', f"{summary['best_asr_clean_acc']:.2%}")
if summary.get('best_tradeoff_acc') is not None:
    print('Best tradeoff ACC:', f"{summary['best_tradeoff_acc']:.2%}")
    print('Best tradeoff ASR:', f"{summary['best_tradeoff_asr']:.2%}")
print('=' * 72)


In [ ]:
# Plot ACC/ASR curves.
import matplotlib.pyplot as plt

epochs = list(range(1, len(history['test_acc']) + 1))
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, [x * 100 for x in history['test_acc']], label='ACC', color='tab:blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Clean ACC')
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, [x * 100 for x in history['test_asr']], label='ASR(single)', color='tab:red')
if history.get('test_asr_multi'):
    plt.plot(epochs, [x * 100 for x in history['test_asr_multi']], label='ASR(multi)', color='tab:orange')
plt.xlabel('Epoch')
plt.ylabel('ASR (%)')
plt.title('Attack Success Rate')
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plot_path = run_dir / 'curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved plot:', plot_path)


In [ ]:
# Optional: package and download run artifacts.
from google.colab import files
import shutil

zip_path = shutil.make_archive(str(run_dir), 'zip', str(run_dir))
print('Created:', zip_path)
files.download(zip_path)
